In [80]:
import pandas as pd

## Analyze the connection between population, permits and zoning across regions ##

**Goal:** Make some connection about whether areas with stricter zoning equates to less permitting. At the end I'll make a connection about how this relates to the Road to Housing Act.

This data analysis takes inspiration from the methodology of Joseph Gyourko, Jonathan S. Hartley and Jacob Krimmel, the authors of the data research.
- This notebook has a section that only includes metro areas for analysis and follows the Wharton Professors method.
- This notebook also uses their method for scoring strictness.

**What I'll analyze:**
- Whether regions have different relationships with zoning and permiting.
- Does population growth equate to more permitting?
- Are any towns that are at risk of losing funding through the Road to Housing Act.

## Load spreadsheet matched with population, permits and strictness

In [81]:
towns = pd.read_csv("../analysis/wharton_permits_matched.csv", dtype={"match_key": str})

print(towns.shape)
towns.head()

(1995, 38)


,GEOID,state,countyname18,communityname18,cbsatitle18,LPPI18,SPII18,CII18,LPAI18,LZAI18,...,units_apartments_avg_2018_22,units_all_avg_2018_22,bps_name,pop_2010,pop_2017,pop_2020,pop_2022,pop_2023,pop_avg_2018_22,permits_per_thousand
0,2502701885,MA,Worcester County,Ashburnham,"Worcester, MA-CT",8.0,2.0,4.0,3.0,3.0,...,0.000000,20.400000,Ashburnham town,6081.0,6308.0,6312.0,6385.0,6415.0,6350.2,3.212497
1,2501702130,MA,Middlesex County,Ashland,"Boston-Cambridge-Newton, MA-NH",11.0,2.0,2.0,3.0,2.0,...,12.800000,28.200000,Ashland town,16593.0,17784.0,18835.0,18572.0,18591.0,18333.0,1.538210
2,2502706365,MA,Worcester County,Bolton,"Worcester, MA-CT",10.0,2.0,2.0,5.0,5.0,...,0.666667,20.333333,Bolton town,4897.0,5283.0,5668.0,5737.0,5738.0,5578.2,3.645142
3,2501768050,MA,Middlesex County,Stow,"Boston-Cambridge-Newton, MA-NH",11.0,1.0,4.0,5.0,2.0,...,0.000000,9.200000,Stow town,6590.0,7083.0,7158.0,7081.0,7088.0,7150.2,1.286677
4,2500538225,MA,Bristol County,Mansfield,"Providence-Warwick, RI-MA",9.0,2.0,3.0,3.0,3.0,...,2.400000,11.600000,Mansfield town,23184.0,23931.0,23862.0,23827.0,23862.0,24027.6,0.482778


## Give towns a score based on their WRLURI18
**The Wharton professor who built this data set made a system of classifying the WRLURI18 score into three categories. In this section I'll recreate this code so I'll be able to analyze strictness in the way the authors believed was best.**
Note: the Wharton professor did this only for metro areas but I will do it for all towns using their methodology.
Highly regulated: Scores in the 75th percentile.
Moderately regulated: Scores in the 26th to 74th percentile.
Low regulated: Socres in the lowest 25th percentile.

In [82]:
#A new column creates 3 categories: Lightly regulated, Moderately regulated and Highly regulated
bottom_percentile=towns["WRLURI18"].quantile(0.25)
top_percentile= towns["WRLURI18"].quantile(0.75)

def strictness(score):
    if score <= bottom_percentile:
        return "Lightly regulated"
    if score >= top_percentile:
        return "Highly regulated"
    return "Moderately regulated"

towns["strictness"] = towns["WRLURI18"].apply(strictness)

towns["strictness"].value_counts()

strictness
Moderately regulated    997
Highly regulated        499
Lightly regulated       499
Name: count, dtype: int64

## Analyze zoning strictness by region
Regions of the country vary greatly in their population growth and zoning strictness, both of which could have an effect on the permits an area might grant. In this section I'll create a dictionary that can be used through the notebook to analyze findings by region.

In [83]:
# Find out if there's a pattern in zoning strictness and region.
# Create a dictionary for the 4 census regions.
region = {
    "CT": "Northeast", "ME": "Northeast", "MA": "Northeast", "NH": "Northeast", "NJ": "Northeast",
    "NY": "Northeast", "PA": "Northeast", "RI": "Northeast", "VT": "Northeast",
    "IL": "Midwest", "IN": "Midwest", "IA": "Midwest", "KS": "Midwest", "MI": "Midwest", "MN": "Midwest",
    "MO": "Midwest", "NE": "Midwest", "ND": "Midwest", "OH": "Midwest", "SD": "Midwest", "WI": "Midwest",
    "AL": "South", "AR": "South", "DE": "South", "FL": "South", "GA": "South", "KY": "South",
    "LA": "South", "MD": "South", "MS": "South", "NC": "South", "OK": "South", "SC": "South",
    "TN": "South", "TX": "South", "VA": "South", "WV": "South", "DC": "South",
    "AK": "West", "AZ": "West", "CA": "West", "CO": "West", "HI": "West", "ID": "West", "MT": "West",
    "NV": "West", "NM": "West", "OR": "West", "UT": "West", "WA": "West", "WY": "West",
}

towns["region"] = towns["state"].apply(region.get)

#Make a table that determines how many places each region has in different strictness categories.
table = pd.crosstab(towns["region"], towns["strictness"])
table = table[sorted(table.columns, reverse=True)]
table

strictness,Moderately regulated,Lightly regulated,Highly regulated
region,,,
Midwest,337,209,80
Northeast,249,124,168
South,245,127,104
West,166,39,147


In [84]:
# Calculate what percentage of towns in each region fall into each regulation category.
pct = table.apply(lambda row: row / row.sum() * 100, axis=1)
pct

strictness,Moderately regulated,Lightly regulated,Highly regulated
region,,,
Midwest,53.833866,33.386581,12.779553
Northeast,46.025878,22.920518,31.053604
South,51.470588,26.680672,21.848739
West,47.159091,11.079545,41.761364


The midwest has the most lightly regulated towns and the west has the most highly regulated towns.

## How many building permits per capita?
The population of a town is a critical factor when analyzing how many permits an area approves. In this section I'll add a column that calculates how many permits each town issues per 1,000 residents.

In [85]:
#Create a table of permits per thousand in each region.
regions_permits = towns.groupby("region").agg({"permits_per_thousand":["mean", "median"]})

regions_permits

permits_per_thousand          
                          mean    median
region                                  
Midwest               3.446200  1.955792
Northeast             2.398680  1.572460
South                 7.501484  4.261424
West                  5.826261  3.873884

In [86]:
#How many permits per thousand in each region and category of strictness.
by_region = towns.groupby(["region","strictness"]).agg(towns=("permits_per_thousand", "mean"),).unstack()
by_region

towns                                       
strictness Highly regulated Lightly regulated Moderately regulated
region                                                            
Midwest            4.237231          2.396208             3.909600
Northeast          2.410774          2.100282             2.539120
South              7.167135          5.391267             8.737280
West               5.595383          4.329058             6.382465

The lightest regulated areas permitted the least housing. This suggests that demand may be a more powerful factor than strictness. I check for growth later in the notebook.

In [87]:
#What regions are the top ten strictest towns in?
towns.groupby(["communityname18", "region"])["WRLURI18"].median().sort_values(ascending=False).head(10)

communityname18  region   
Kingstree        South        4.861689
Menands          Northeast    3.688969
Niagara          Northeast    3.500069
Rush             Northeast    3.455736
Coral Springs    South        3.353712
Ridgewood        Northeast    3.203057
Mineola          South        3.198566
Calabasas        West         3.067194
Annapolis        South        3.031375
Laguna Beach     West         2.989217
Name: WRLURI18, dtype: float64

 Three out of the top 5 towns with strictest zoning are in the Northeast.

## Group all the metro areas
The Wharton professors grouped metro areas because they say they function as housing markets more accurately than individual towns and therefore work better for analysis. I wanted to create code that recreated their list of metro areas so I could do analysis on both metro areas and towns.

In [88]:
#Create a filtered table that drops metro areas with less than 10 places.
counts = towns["cbsatitle18"].value_counts()
metro_towns =towns[towns["cbsatitle18"].isin(counts[counts >= 10].index)]

#Find the mean WRLURI18 of each metro area with 10 or more places and keep the state and permits_per_thousand column.
#Note: some of the metro's state column leave out a state. That is because none of towns in that state answered the survey.
metro_scores = metro_towns.groupby("cbsatitle18").agg(
    WRLURI18=("WRLURI18", "mean"),
    permits_per_thousand=("permits_per_thousand","mean"),
    states=("state", "unique"),
)

#Convert state column to text.
metro_scores["states"] = metro_scores["states"].str.join(",")

metro_scores.head(20)


,WRLURI18,permits_per_thousand,states
cbsatitle18,,,
"Albany-Schenectady-Troy, NY",0.474240,3.210136,NY
"Allentown-Bethlehem-Easton, PA-NJ",0.129801,2.615559,PA
"Atlanta-Sandy Springs-Roswell, GA",-0.118132,10.167911,GA
"Boston-Cambridge-Newton, MA-NH",0.304619,3.101190,"MA,NH"
"Buffalo-Cheektowaga-Niagara Falls, NY",0.054719,1.035531,NY
"Chicago-Naperville-Elgin, IL-IN-WI",-0.125804,2.588120,"IN,IL,WI"
"Cincinnati, OH-KY-IN",-0.228901,2.210257,"KY,OH"
"Cleveland-Elyria, OH",-0.732844,1.217161,OH
"Dallas-Fort Worth-Arlington, TX",0.169008,12.323744,TX


In [89]:
#In table three of the Wharton Professor's paper they share how they broke up the WRLURI18 scores into categories.
#I'm using their number since my metro areas match theirs.
paper_bottom = -0.642
paper_top = 0.637

def paper_strictness(score):
    if score <= paper_bottom:
        return "Lightly regulated"
    if score >= paper_top:
        return "Highly regulated"
    return "Moderately regulated"

metro_scores["paper_strictness"] = metro_scores["WRLURI18"].apply(paper_strictness)

metro_scores["paper_strictness"].value_counts()


paper_strictness
Moderately regulated    29
Highly regulated         7
Lightly regulated        1
Name: count, dtype: int64

## Analyze Metro Zoning
Most of this is the same analysis that I did for all the towns above.

In [90]:
#Sort the metros in order of most to least strict zoning.
metro_scores = metro_scores.sort_values("WRLURI18", ascending=False)
metro_scores.head(5)

,WRLURI18,permits_per_thousand,states,paper_strictness
cbsatitle18,,,,
"San Francisco-Oakland-Hayward, CA",1.169552,2.636632,CA,Highly regulated
"New York-Newark-Jersey City, NY-NJ-PA",1.090671,3.233053,"NJ,NY,PA",Highly regulated
"Providence-Warwick, RI-MA",0.928382,2.130674,"MA,RI",Highly regulated
"Los Angeles-Long Beach-Anaheim, CA",0.750458,1.225484,CA,Highly regulated
"Seattle-Tacoma-Bellevue, WA",0.723992,6.253176,WA,Highly regulated


In [91]:
# Find out if there's a pattern in zoning strictness and region.
# Create a dictionary for the 4 census regions.
region = {
    "CT": "Northeast", "ME": "Northeast", "MA": "Northeast", "NH": "Northeast", "NJ": "Northeast",
    "NY": "Northeast", "PA": "Northeast", "RI": "Northeast", "VT": "Northeast",
    "IL": "Midwest", "IN": "Midwest", "IA": "Midwest", "KS": "Midwest", "MI": "Midwest", "MN": "Midwest",
    "MO": "Midwest", "NE": "Midwest", "ND": "Midwest", "OH": "Midwest", "SD": "Midwest", "WI": "Midwest",
    "AL": "South", "AR": "South", "DE": "South", "FL": "South", "GA": "South", "KY": "South",
    "LA": "South", "MD": "South", "MS": "South", "NC": "South", "OK": "South", "SC": "South",
    "TN": "South", "TX": "South", "VA": "South", "WV": "South", "DC": "South",
    "AK": "West", "AZ": "West", "CA": "West", "CO": "West", "HI": "West", "ID": "West", "MT": "West",
    "NV": "West", "NM": "West", "OR": "West", "UT": "West", "WA": "West", "WY": "West","MA,NH": "Northeast", 
    "IN,IL,WI": "Midwest", "KY,OH": "Midwest", "KS,MO": "Midwest", "MN,WI": "Midwest", "NJ,NY,PA": "Northeast",
    "NJ,PA": "Northeast", "OR,WA": "West", "IL,MO": "Midwest", "MA,CT": "Northeast", "MA,RI": "Northeast"
}

metro_scores["region"] = metro_scores["states"].apply(region.get)

#Make a table that determines how many places each region has in different strictness categories.
table = pd.crosstab(metro_scores["region"], metro_scores["paper_strictness"])
table = table[sorted(table.columns, reverse=True)]
table

paper_strictness,Moderately regulated,Lightly regulated,Highly regulated
region,,,
Midwest,9,1,0
Northeast,13,0,2
South,5,0,0
West,2,0,5


In [92]:
#Only metros that have both permit and population data and call it regional_permits_strictness.
metro_regional_permits_strictness = metro_scores.dropna(subset=["permits_per_thousand"])

#How many permits per thousand in each region and category of strictness.
by_region = metro_regional_permits_strictness.groupby(["region","paper_strictness"]).agg(metros=("permits_per_thousand", "mean"),).unstack()
by_region

metros                                       
paper_strictness Highly regulated Lightly regulated Moderately regulated
region                                                                  
Midwest                       NaN          1.217161             4.995214
Northeast                2.681863               NaN             2.446190
South                         NaN               NaN             8.419799
West                     5.527274               NaN             9.121843

The data in this table is a little thin -- not all regions have more than one metro in a category. Sometimes the means are calculated based off of very few towns.

In [93]:
#What regions are the top ten strictest metros in?
metro_scores.groupby(["cbsatitle18", "region"])["WRLURI18"].median().sort_values(ascending=False).head(10)

cbsatitle18                                region   
San Francisco-Oakland-Hayward, CA          West         1.169552
New York-Newark-Jersey City, NY-NJ-PA      Northeast    1.090671
Providence-Warwick, RI-MA                  Northeast    0.928382
Los Angeles-Long Beach-Anaheim, CA         West         0.750458
Seattle-Tacoma-Bellevue, WA                West         0.723992
Riverside-San Bernardino-Ontario, CA       West         0.663325
Phoenix-Mesa-Scottsdale, AZ                West         0.640515
Portland-Vancouver-Hillsboro, OR-WA        West         0.631992
Miami-Fort Lauderdale-West Palm Beach, FL  South        0.627314
Albany-Schenectady-Troy, NY                Northeast    0.474240
Name: WRLURI18, dtype: float64

Six out of the top ten metro areas are in the West.

## Does population affect this?
In this section I want to do further analysis about how growth in a city affects permitting.
- If a city is growing do they issue more permits?
- How the fastest growing towns are split across strictness categories.

In [94]:
# Out of the top 25% percentile of towns that grew between 2010 and 2017, how many were in each strictness category?
towns["growth_2010_2017"]= (towns["pop_2017"] - towns["pop_2010"])/ towns["pop_2010"]

cut_off= towns["growth_2010_2017"].quantile(0.75)
fast_growth= towns[towns["growth_2010_2017"]>= cut_off]

fast_growth["strictness"].value_counts()

strictness
Moderately regulated    276
Highly regulated        160
Lightly regulated        62
Name: count, dtype: int64

In [95]:
# Create a table that determines how many permits per thousand cities that grew got vs. cities that didn't.

towns["grew"] = towns["growth_2010_2017"] > 0

towns.groupby("grew").agg(
    permits_per_thousand=("permits_per_thousand", "mean"),
)

,permits_per_thousand
grew,
False,1.754301
True,5.848814


In [96]:
#Find the percentage of town that grew in each category of strictness.
pd.crosstab(towns["grew"], towns["strictness"], normalize="index")* 100


strictness,Highly regulated,Lightly regulated,Moderately regulated
grew,,,
False,19.431280,39.494471,41.074250
True,27.606461,18.281938,54.111601


Growing towns permitted about three times as much. These towns were more likely to be highly regulated.

## What cities will the ROAD to Housing act affect?

The legislation could take away CDBG federal grants for cities who are underbuilding housing. CDBG money helps with home repairs, public infrastructure and community services.

- The act could take away 10% of CDBG funding from anyone who improved building below the median. This calculated by comparing the most recent five years to the 5 years before.
- This codes examines what towns this could possibly be.

**Caveats**
(This is intended to be a **very** rough predictor for personal analysis.)
- This uses permit data, not houses actually built, so more cities could be at_risk if they didn't actually end up building the houses.
- Some of these towns would be automatically out of the running to lose funding if they had a 4% increase of their total housing stock. I don't have the stats for total housing stock so I didn't calculate it. This has the potential to change the results significantly.

In [97]:
#Load the data set for 2021-2025
recent = pd.read_csv("../analysis/wharton_permits_matched_2021_2025.csv", dtype={"match_key": str})


In [98]:
# Convert full state names to the two-letter abbreviations used in the region dictionary.
#Add a region column to the recent data set.
state_abbr = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
    "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "District of Columbia": "DC",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL",
    "Indiana": "IN", "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA",
    "Maine": "ME", "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK", "Oregon": "OR",
    "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC", "South Dakota": "SD",
    "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT", "Virginia": "VA",
    "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
}

recent["state"] = recent["STNAME"].map(state_abbr)
recent["region"] = recent["state"].map(region)

recent["region"].isna().sum()


np.int64(0)

In [ ]:
#CDBG is only given to places with populations of above 50,000. Drop others from data frame.
big = recent[recent["POPESTIMATE2025"] >= 50000]

#The act compares building in the last 5 years (2021-2025) with the 5 years before (2016-2020).
new = big["units_all_avg_2021_25"]
old = big["units_all_avg_2016_20"]
big["improvement"] = (new - old) / (new + old) # Found in Section 213(a)(2) and Section 213(a)(5) of law.

#Drop places without 2016-2020 data, since they can't get a score.
big = big.dropna(subset=["improvement"])

#Find the median improvement score.
median = big["improvement"].median()

#Create a data frame with median improvement score of towns.
#Keep only bps_name, region, two permit averages, improvement score and permits_per_thousand_2025 column.
at_risk=big[big["improvement"] < median]
at_risk=at_risk.sort_values("improvement")[["bps_name", "region", "permits_per_thousand_2025", "units_all_avg_2016_20",
                                              "units_all_avg_2021_25", "improvement"]]

at_risk

,bps_name,region,permits_per_thousand_2025,units_all_avg_2016_20,units_all_avg_2021_25,improvement
1563,Sunrise,South,0.008158,77.0,0.800000,-0.979434
1360,Coconut Creek,South,0.020653,90.4,1.200000,-0.973799
4573,Medford,Northeast,0.277335,899.5,16.500000,-0.963974
612,Costa Mesa,West,0.070707,290.0,7.750000,-0.947943
2166,Des Plaines,Midwest,0.127466,190.8,7.500000,-0.924357
...,...,...,...,...,...,...
12974,Laredo,South,5.717823,1382.6,1499.333333,0.040505
8963,Wake Forest town,South,15.673555,786.8,853.400000,0.040605
13674,Newport News,South,1.390194,235.0,255.400000,0.041599
4866,Dearborn Heights,Midwest,0.202641,11.4,12.400000,0.042017


In [100]:
#Find out how many place in each region are at risk of losing funding.
at_risk["region"].value_counts()

region
West         137
South        126
Midwest       81
Northeast     57
Name: count, dtype: int64

In [101]:
#Create a data frame with places that are above the median improvement score.
#Keep only bps_name, region and improvement column.
not_at_risk=big[big["improvement"] >= median]
not_at_risk=not_at_risk.sort_values("improvement", ascending=False)[["bps_name", "region", "improvement"]]

not_at_risk

,bps_name,region,improvement
11019,Lancaster,Northeast,0.958922
14818,Racine,Midwest,0.949527
8207,Hempstead village,Northeast,0.885117
6418,Florissant,Midwest,0.883721
7967,Cheektowaga town,Northeast,0.873874
...,...,...,...
12503,Nashville-Davidson,South,0.046859
13304,Waco,South,0.046568
13675,Norfolk,South,0.045470
6200,Jackson,South,0.045238


Three out of the top five places with the highest improvement scores are in the Northeast.

In [102]:
#Find out how many place in each region are not at risk of losing funding.
not_at_risk["region"].value_counts()

region
West         142
South        115
Midwest       78
Northeast     66
Name: count, dtype: int64

In [103]:
#Percentage of places at risk of losing funding in each region.
perc_at_risk= (at_risk["region"].value_counts() / big["region"].value_counts() * 100)

perc_at_risk

region
West         49.103943
South        52.282158
Midwest      50.943396
Northeast    46.341463
Name: count, dtype: float64

In [104]:
#Percentage of places not at risk of losing funding in each region.
perc_not_at_risk= (not_at_risk["region"].value_counts() / big["region"].value_counts() * 100)

perc_not_at_risk


region
West         50.896057
South        47.717842
Midwest      49.056604
Northeast    53.658537
Name: count, dtype: float64

In [105]:
#Were the places with high improvement scores building very little before 2021?
#Make a permits per thousand column for 2016-2020 so places of different sizes can be compared.
big["permits_per_thousand_2016_20"] = big["units_all_avg_2016_20"] / big["pop_avg_2021_25"] * 1000

#The 50 places with the highest improvement scores and the 50 places with the lowest.
top = big.sort_values("improvement", ascending=False).head(50)
bottom = big.sort_values("improvement").head(50)

#Compare the median permits per thousand before and after 2021 for each group with all places.
["permits_per_thousand_2016_20", "permits_per_thousand_2025"]
pd.DataFrame({
    "highest 50": top[["permits_per_thousand_2016_20", "permits_per_thousand_2025"]].median(),
    "all places": big[["permits_per_thousand_2016_20", "permits_per_thousand_2025"]].median(),
    "lowest 50": bottom[["permits_per_thousand_2016_20", "permits_per_thousand_2025"]].median(),
})


,highest 50,all places,lowest 50
permits_per_thousand_2016_20,0.502782,3.251659,3.08913
permits_per_thousand_2025,2.434979,3.137449,0.55045


The South is marginally more at risk and the Northeast is marginally less at risk. Because the law targets cities that had less of an increase (or none) in building in the last five years, for many of the places, their high improvement score is a result of them building very little before. It doesn't necessarily indicate that they are building a lot now. In fact, the places with the highest improvement scores built less in the last five years than the places with the lowest improvement scores did in the five years before that.